In [58]:
import pandas as pd
import numpy as np 
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoNormal
import pyro.optim as optim


## Preprocessing

In [ ]:
# import dataset
df = pd.read_csv("Cleaned_dataset.csv")

In [60]:
# print features
print(df.columns)

Index(['Date_of_journey', 'Journey_day', 'Airline', 'Flight_code', 'Class',
       'Source', 'Departure', 'Total_stops', 'Arrival', 'Destination',
       'Duration_in_hours', 'Days_left', 'Fare'],
      dtype='object')


In [ ]:
# select the features we want to use for the model
df = df[["Journey_day", "Airline", "Source", "Departure", "Total_stops",
         "Arrival", "Destination", "Duration_in_hours", "Days_left", "Fare"]]

# convert fare currency from INR to USD
df[["Fare"]] = (df[["Fare"]] * 0.01066).round(0).astype("Int64")

print(df.dtypes)

Journey_day           object
Airline               object
Source                object
Departure             object
Total_stops           object
Arrival               object
Destination           object
Duration_in_hours    float64
Days_left              int64
Fare                   Int64
dtype: object


In [ ]:
# use integer encoding for categorical features
df["Journey_day"] = df["Journey_day"].astype("category").cat.codes
df["Airline"] = df["Airline"].astype("category").cat.codes
df["Source"] = df["Source"].astype("category").cat.codes
df["Departure"] = df["Departure"].astype("category").cat.codes
df["Total_stops"] = df["Total_stops"].astype("category").cat.codes
df["Arrival"] = df["Arrival"].astype("category").cat.codes
df["Destination"] = df["Destination"].astype("category").cat.codes

# list of the number of unique values for each categorical variable
cardinalities = [len(df["Journey_day"].unique()), len(df["Airline"].unique()), len(df["Source"].unique()), len(df["Departure"].unique()),
        len(df["Total_stops"].unique()), len(df["Arrival"].unique()), len(df["Destination"].unique())]


# split into training and test set
train_fraction = 0.8
df_shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

split_idx = int(len(df_shuffled) * train_fraction)

df_train = df_shuffled.iloc[:split_idx].copy()
df_test = df_shuffled.iloc[split_idx:].copy()

# compute scaling to standardize numerical features based on the training set only
duration_mean = df_train["Duration_in_hours"].mean()
duration_std = df_train["Duration_in_hours"].std()
days_left_mean = df_train["Days_left"].mean()
days_left_std = df_train["Days_left"].std()

# compute scaling for target variable (fare)
fare_mean = df_train["Fare"].mean()
fare_std = df_train["Fare"].std()

# apply scaling to training set
df_train["Duration_in_hours"] = (df_train["Duration_in_hours"] - duration_mean) / duration_std
df_train["Days_left"] = (df_train["Days_left"] - days_left_mean) / days_left_std
df_train["Fare"] = (df_train["Fare"] - fare_mean) / fare_std

# apply scaling to test set
df_test["Duration_in_hours"] = (df_test["Duration_in_hours"] - duration_mean) / duration_std
df_test["Days_left"] = (df_test["Days_left"] - days_left_mean) / days_left_std
df_test["Fare"] = (df_test["Fare"] - fare_mean) / fare_std


# create torch tensors for each categorical variable for both training and test set
journey_day_tensor_train = torch.tensor(df_train["Journey_day"].values, dtype=torch.long)
journey_day_tensor_test = torch.tensor(df_test["Journey_day"].values, dtype=torch.long)
airline_tensor_train = torch.tensor(df_train["Airline"].values, dtype=torch.long)
airline_tensor_test = torch.tensor(df_test["Airline"].values, dtype=torch.long)
source_tensor_train = torch.tensor(df_train["Source"].values, dtype=torch.long)
source_tensor_test = torch.tensor(df_test["Source"].values, dtype=torch.long)
departure_tensor_train = torch.tensor(df_train["Departure"].values, dtype=torch.long)
departure_tensor_test = torch.tensor(df_test["Departure"].values, dtype=torch.long)
total_stops_tensor_train = torch.tensor(df_train["Total_stops"].values, dtype=torch.long)
total_stops_tensor_test = torch.tensor(df_test["Total_stops"].values, dtype=torch.long)
arrival_tensor_train = torch.tensor(df_train["Arrival"].values, dtype=torch.long)
arrival_tensor_test = torch.tensor(df_test["Arrival"].values, dtype=torch.long)
destination_tensor_train = torch.tensor(df_train["Destination"].values, dtype=torch.long)
destination_tensor_test = torch.tensor(df_test["Destination"].values, dtype=torch.long)

# create torch tensors for each numerical variable for both training and test set
duration_in_hours_tensor_train = torch.tensor(df_train["Duration_in_hours"].values, dtype=torch.float)
duration_in_hours_tensor_test = torch.tensor(df_test["Duration_in_hours"].values, dtype=torch.float)
days_left_tensor_train = torch.tensor(df_train["Days_left"].values, dtype=torch.float)
days_left_tensor_test = torch.tensor(df_test["Days_left"].values, dtype=torch.float)

# create torch tensor for target variable for both training and test set
fare_tensor_train = torch.tensor(df_train["Fare"].values, dtype=torch.float)
fare_tensor_test = torch.tensor(df_test["Fare"].values, dtype=torch.float)


# lists of each type of feature (categorical and numerical)
cat_tensors_train = [journey_day_tensor_train, airline_tensor_train, source_tensor_train, departure_tensor_train, total_stops_tensor_train,
                     arrival_tensor_train, destination_tensor_train]
cat_tensors_test = [journey_day_tensor_test, airline_tensor_test, source_tensor_test, departure_tensor_test, total_stops_tensor_test,
                     arrival_tensor_test, destination_tensor_test]
num_tensors_train = [duration_in_hours_tensor_train, days_left_tensor_train]
num_tensors_test = [duration_in_hours_tensor_test, days_left_tensor_test]


## Define model

In [ ]:
# bayesian hierarchical model

def flight_model(cat_tensors, num_tensors, cardinalities, fare=None):

    n_obs = len(num_tensors[0]) # total number of flights
    
    # since "fare" is standardized (mean 0, std 1), the global intercept should be near 0
    intercept = pyro.sample("global_intercept", dist.Normal(0.0, 1.0))
    
    # dynamic hierarchical plates for categorical variables
    # loop through the 7 categorical features to learn their unique baselines
    mu_categorical = 0.0
    for i in range(len(cat_tensors)):
        # each category gets its own learned variance
        sigma_cat = pyro.sample(f"sigma_cat_{i}", dist.HalfCauchy(1.0))
        
        with pyro.plate(f"plate_cat_{i}", cardinalities[i]):
            # baseline offset for this specific category
            alpha = pyro.sample(f"alpha_cat_{i}", dist.Normal(0.0, sigma_cat))
            
        # add the mapped intercepts to our linear predictor
        mu_categorical += alpha[cat_tensors[i]]
        
    # fixed effects for numerical variables
    # since data is standardized, a slope of 1.0 is a massive effect. Normal(0, 1) is a good weakly-informative prior
    beta_0 = pyro.sample("beta_num_0", dist.Normal(0.0, 1.0))
    beta_1 = pyro.sample("beta_num_1", dist.Normal(0.0, 1.0))
    
    mu_numerical = (beta_0 * num_tensors[0]) + (beta_1 * num_tensors[1])
    
    # the final linear predictor
    mu = intercept + mu_categorical + mu_numerical
    
    # the likelihood (observation model)
    sigma_obs = pyro.sample("sigma_obs", dist.HalfCauchy(1.0))
    
    with pyro.plate("data", n_obs):
        pyro.sample("obs", dist.Normal(mu, sigma_obs), obs=fare)

## Training model with stochastic variational inference

In [ ]:
pyro.clear_param_store()

# generate a guide for the model
guide = AutoNormal(flight_model)


# setup the Adam optimizer and the SVI inference algorithm
adam_args = {"lr": 0.05} # Learning rate
optimizer = optim.Adam(adam_args)

svi = SVI(
    model=flight_model,
    guide=guide,
    optim=optimizer,
    loss=Trace_ELBO()
)


# training loop
num_epochs = 2000
loss_history = []

print("Starting training...")
for epoch in range(num_epochs):
    # calculate the loss and take a gradient step
    loss = svi.step(cat_tensors_train, num_tensors_train, cardinalities, fare=fare_tensor_train)
    
    # normalize the loss by the number of data points
    loss_history.append(loss / len(fare_tensor_train))
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Loss: {loss / len(fare_tensor_train):.4f}")

print("Training complete!")

Starting training...
Epoch 0 | Loss: 1.4958
Epoch 200 | Loss: 1.3064
Epoch 400 | Loss: 1.3033
Epoch 600 | Loss: 1.3007
Epoch 800 | Loss: 1.3013
Epoch 1000 | Loss: 1.3011
Epoch 1200 | Loss: 1.3004
Epoch 1400 | Loss: 1.3015
Epoch 1600 | Loss: 1.3003
Epoch 1800 | Loss: 1.3008
Training complete!


In [ ]:
# median values for all parameters
predicted_params = guide.median()

# the learned slope for the duration feature as an example
beta_0_estimate = predicted_params["beta_num_0"].item()
print(f"Learned beta for numerical feature 0: {beta_0_estimate:.4f}")

# to see the uncertainty (e.g., the 90% credible interval)
quantiles = guide.quantiles([0.05, 0.95])
lower_bound = quantiles["beta_num_0"][0].item()
upper_bound = quantiles["beta_num_0"][1].item()
print(f"With 90% confidence the true beta is between {lower_bound:.4f} and {upper_bound:.4f}")

Learned beta for numerical feature 0: -0.0593
With 90% confidence the true beta is between -0.0634 and -0.0553


## Testing model

In [ ]:
# draw 1,000 samples from the learned distributions
predictive = Predictive(
    model=flight_model, 
    guide=guide, 
    num_samples=1000, 
    return_sites=("obs",) # This tells Pyro to grab the output of your likelihood plate
)

# run the test data through the predictive model
# we pass price=None so the model generates new prices rather than conditioning on them
print("Generating predictions for the test set...")
samples = predictive(
    cat_tensors_test, 
    num_tensors_test, 
    cardinalities, 
    fare=None
)

# extract the predictions tensor
# this will have the shape: (1000 samples, N test observations)
pred_price_samples = samples["obs"]

Generating predictions for the test set...


In [55]:
# calculate the Median Prediction (Point Estimate)
pred_median = torch.median(pred_price_samples, dim=0).values

# calculate the 90% Prediction Interval (Uncertainty Bounds)
pred_lower = torch.quantile(pred_price_samples, 0.05, dim=0)
pred_upper = torch.quantile(pred_price_samples, 0.95, dim=0)


# --- DETERMINISTIC METRICS ---
# Mean Absolute Error (MAE) on the scaled data
mae_scaled = torch.mean(torch.abs(pred_median - fare_tensor_test)).item()

# Root Mean Squared Error (RMSE) on the scaled data
rmse_scaled = torch.sqrt(torch.mean((pred_median - fare_tensor_test)**2)).item()

# --- PROBABILISTIC METRICS (Coverage) ---
# check if the true price falls within the 90% interval
in_bounds = (fare_tensor_test >= pred_lower) & (fare_tensor_test <= pred_upper)

# calculate the percentage of test points that fell inside the bounds
coverage = in_bounds.float().mean().item() * 100

print(f"--- Scaled Metrics ---")
print(f"MAE (Scaled):  {mae_scaled:.4f}")
print(f"RMSE (Scaled): {rmse_scaled:.4f}")
print(f"90% Interval Coverage: {coverage:.2f}%")

--- Scaled Metrics ---
MAE (Scaled):  0.7101
RMSE (Scaled): 0.8862
90% Interval Coverage: 92.61%


In [64]:
# un-scale the MAE and RMSE
mae_dollars = mae_scaled * fare_std
rmse_dollars = rmse_scaled * fare_std

print(f"--- Real-World Metrics ---")
print(f"Mean Absolute Error: ${mae_dollars:.2f}")
print(f"Root Mean Squared Error: ${rmse_dollars:.2f}")

--- Real-World Metrics ---
Mean Absolute Error: $153.83
Root Mean Squared Error: $191.98
